In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [2]:
credentials_location = '/home/codespace/.config/gcloud/application_default_credentials.json'

gcs_jar = '/workspaces/etl-zoomcamp-project/lib/gcs-connector-hadoop3-2.2.5.jar'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('my-etl-project') \
    .set("spark.jars", gcs_jar) \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [3]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

26/03/22 17:48:47 WARN Utils: Your hostname, codespaces-b411cf resolves to a loopback address: 127.0.0.1; using 10.0.1.104 instead (on interface eth0)
26/03/22 17:48:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/22 17:48:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 17:48:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/22 17:48:50 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [5]:
df = spark.read.csv("gs://mental-health-etl-bucket/raw/mental_health_data.csv", header=True)

In [6]:
df.show()

+-----+-----------+----+----+-----------------+--------------------+--------------------+---------------------+----------------------+--------------+-------------------------+
|index|     Entity|Code|Year|Schizophrenia (%)|Bipolar disorder (%)|Eating disorders (%)|Anxiety disorders (%)|Drug use disorders (%)|Depression (%)|Alcohol use disorders (%)|
+-----+-----------+----+----+-----------------+--------------------+--------------------+---------------------+----------------------+--------------+-------------------------+
|    0|Afghanistan| AFG|1990|          0.16056|            0.697779|            0.101855|              4.82883|              1.677082|      4.071831|                 0.672404|
|    1|Afghanistan| AFG|1991|         0.160312|            0.697961|            0.099313|              4.82974|              1.684746|      4.079531|                 0.671768|
|    2|Afghanistan| AFG|1992|         0.160135|            0.698107|            0.096692|             4.831108|         

In [7]:
df.printSchema()

root
 |-- index: string (nullable = true)
 |-- Entity: string (nullable = true)
 |-- Code: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Schizophrenia (%): string (nullable = true)
 |-- Bipolar disorder (%): string (nullable = true)
 |-- Eating disorders (%): string (nullable = true)
 |-- Anxiety disorders (%): string (nullable = true)
 |-- Drug use disorders (%): string (nullable = true)
 |-- Depression (%): string (nullable = true)
 |-- Alcohol use disorders (%): string (nullable = true)



In [22]:
from pyspark.sql.functions import sum, when

df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]).show()

[Stage 26:=============================>                            (1 + 1) / 2]

+-----+------+----+----+-----------------+--------------------+--------------------+---------------------+----------------------+--------------+-------------------------+
|index|Entity|Code|Year|Schizophrenia (%)|Bipolar disorder (%)|Eating disorders (%)|Anxiety disorders (%)|Drug use disorders (%)|Depression (%)|Alcohol use disorders (%)|
+-----+------+----+----+-----------------+--------------------+--------------------+---------------------+----------------------+--------------+-------------------------+
|    0|     0|5412| 143|            82681|               89149|                8319|               102085|                102085|        102085|                   102085|
+-----+------+----+----+-----------------+--------------------+--------------------+---------------------+----------------------+--------------+-------------------------+



In [23]:
df.count()

108553

In [9]:
from pyspark.sql.functions import col, count

df.groupBy(df.columns).count().filter(col("count") > 1).show()

+-----+------+----+----+-----------------+--------------------+--------------------+---------------------+----------------------+--------------+-------------------------+-----+
|index|Entity|Code|Year|Schizophrenia (%)|Bipolar disorder (%)|Eating disorders (%)|Anxiety disorders (%)|Drug use disorders (%)|Depression (%)|Alcohol use disorders (%)|count|
+-----+------+----+----+-----------------+--------------------+--------------------+---------------------+----------------------+--------------+-------------------------+-----+
+-----+------+----+----+-----------------+--------------------+--------------------+---------------------+----------------------+--------------+-------------------------+-----+



In [10]:
from pyspark.sql import  types
from pyspark.sql.functions import to_timestamp, year, coalesce, lit


schema = types.StructType([
    types.StructField('index', types.StringType(), True),
    types.StructField('Entity', types.StringType(), True),
    types.StructField('Code', types.StringType(), True),
    types.StructField('Year', types.IntegerType(), True),

    types.StructField('Schizophrenia (%)', types.FloatType(), True),
    types.StructField('Bipolar disorder (%)', types.FloatType(), True),
    types.StructField('Eating disorders (%)', types.FloatType(), True),
    types.StructField('Anxiety disorders (%)', types.FloatType(), True),
    types.StructField('Drug use disorders (%)', types.FloatType(), True),
    types.StructField('Depression (%)', types.FloatType(), True),
    types.StructField('Alcohol use disorders (%)', types.FloatType(), True)
])

df = spark.read.csv("gs://mental-health-etl-bucket/raw/mental_health_data.csv", header=True, schema=schema)

In [11]:
df.printSchema()

root
 |-- index: string (nullable = true)
 |-- Entity: string (nullable = true)
 |-- Code: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Schizophrenia (%): float (nullable = true)
 |-- Bipolar disorder (%): float (nullable = true)
 |-- Eating disorders (%): float (nullable = true)
 |-- Anxiety disorders (%): float (nullable = true)
 |-- Drug use disorders (%): float (nullable = true)
 |-- Depression (%): float (nullable = true)
 |-- Alcohol use disorders (%): float (nullable = true)



In [12]:
entities = [row["Entity"] for row in df.select("Entity").distinct().collect()]
print(entities)

[Stage 10:=============================>                            (1 + 1) / 2]

['South Asia', 'Chad', 'Paraguay', 'Russia', 'Anguilla', 'Macao', 'Southeast Asia, East Asia, and Oceania', 'World', 'Yemen', 'Senegal', 'Sweden', 'Tokelau', 'Kiribati', 'Guyana', 'Eritrea', 'Philippines', 'Djibouti', 'High-income Asia Pacific', 'Tonga', 'Latin America', 'High-income', 'Malaysia', 'Singapore', 'Fiji', 'Turkey', 'United States Virgin Islands', 'Malawi', 'Western Sahara', 'Iraq', 'Sint Maarten (Dutch part)', 'Eastern Europe', 'Germany', 'Northern Mariana Islands', 'Europe', 'Comoros', 'Afghanistan', 'Cambodia', 'Jordan', 'Maldives', 'Rwanda', 'Saint Helena', 'Sudan', 'Palau', 'France', 'Turks and Caicos Islands', 'Greece', 'Sri Lanka', 'Africa', 'Montserrat', 'Dominica', 'Taiwan', 'British Virgin Islands', 'Algeria', 'Equatorial Guinea', 'Togo', 'Slovakia', 'Tropical Latin America', 'Reunion', 'Argentina', 'Wales', 'Angola', 'Belgium', 'Vatican', 'San Marino', 'Middle SDI', 'Ecuador', 'High SDI', 'Qatar', 'Lesotho', 'Bonaire Sint Eustatius and Saba', 'Albania', 'Madagasc

In [13]:
from pyspark.sql.functions import col, round, when

percent_columns = [
    "Schizophrenia (%)",
    "Bipolar disorder (%)",
    "Eating disorders (%)",
    "Anxiety disorders (%)",
    "Drug use disorders (%)",
    "Depression (%)",
    "Alcohol use disorders (%)"
]

for c in percent_columns:
    df = df.withColumn(
        c,
        when(col(c).isNotNull(), round(col(c).cast("float"), 2)).otherwise(None)
    )

In [14]:
df.show()

+-----+-----------+----+----+-----------------+--------------------+--------------------+---------------------+----------------------+--------------+-------------------------+
|index|     Entity|Code|Year|Schizophrenia (%)|Bipolar disorder (%)|Eating disorders (%)|Anxiety disorders (%)|Drug use disorders (%)|Depression (%)|Alcohol use disorders (%)|
+-----+-----------+----+----+-----------------+--------------------+--------------------+---------------------+----------------------+--------------+-------------------------+
|    0|Afghanistan| AFG|1990|             0.16|                 0.7|                 0.1|                 4.83|                  1.68|          4.07|                     0.67|
|    1|Afghanistan| AFG|1991|             0.16|                 0.7|                 0.1|                 4.83|                  1.68|          4.08|                     0.67|
|    2|Afghanistan| AFG|1992|             0.16|                 0.7|                 0.1|                 4.83|         

In [15]:
df = df.dropDuplicates()

In [16]:
from pyspark.sql.functions import col


columns_to_keep = [
    "Entity",
    "Year",
    "Bipolar disorder (%)",
    "Eating disorders (%)",
    "Anxiety disorders (%)",
    "Drug use disorders (%)",
    "Depression (%)",
    "Alcohol use disorders (%)"
]

df_selected = df.select([col(c).alias(c.replace(" (%)", "")) for c in columns_to_keep])

In [17]:
df_selected = df_selected.withColumnRenamed("Entity", "country") \
       .withColumnRenamed("Year", "year") \
       .withColumnRenamed("Bipolar disorder", "bipolar_disorder") \
       .withColumnRenamed("Eating disorders", "eating_disorders") \
       .withColumnRenamed("Anxiety disorders", "anxiety_disorders") \
       .withColumnRenamed("Drug use disorders", "drug_use_disorders") \
       .withColumnRenamed("Depression", "depression") \
       .withColumnRenamed("Alcohol use disorders", "alcohol_use_disorders")

In [18]:
df_selected.show()

[Stage 14:=============================>                            (1 + 1) / 2]

+------------+----+----------------+----------------+-----------------+------------------+----------+---------------------+
|     country|year|bipolar_disorder|eating_disorders|anxiety_disorders|drug_use_disorders|depression|alcohol_use_disorders|
+------------+----+----------------+----------------+-----------------+------------------+----------+---------------------+
| Afghanistan|1999|             0.7|            0.09|             4.83|              1.77|      4.12|                 0.66|
| Afghanistan|2016|            0.71|            0.11|             4.88|              2.51|      4.14|                 0.66|
|  Azerbaijan|1990|            0.68|            0.17|             2.58|              0.45|       2.6|                 2.12|
|     Belgium|1992|            0.96|            0.52|             5.23|              0.83|      3.63|                 1.42|
|       Congo|1999|            0.61|            0.14|             3.26|              0.61|      4.26|                 1.55|
|      F

In [20]:
df_selected.write.mode("overwrite").parquet("gs://mental-health-etl-bucket/processed/mental_health_data.parquet")